# Lab 04 — Windowing & the leakage↔resolution trade

> **Type-2 lab — student version.** Fill in each `# TODO` (the answer cells raise `NotImplementedError` until you do), answer the **Checkpoint** questions, and complete the reflection cell, then re-run top to bottom. Instructor solutions are not distributed in this repository.

**Covers.** Chapter 4 — §4.2–§4.5 (leakage, window functions, window properties, the resolution↔leakage trade), §4.6 (selection guidelines).

**Biomedical question.** Which window lets me see the peaks I care about — without leakage burying a weak one?
**Task type (§1.8).** Representation (spectral) — choosing the window that preserves the peaks the question depends on.
**Information that must be preserved.** two things one window cannot both guarantee — the ability to tell **close peaks apart** (resolution) AND to see a **weak peak beside a strong one** (dynamic range).
**Main assumptions.** a finite 1 s record; tones stationary over it; the spectrum is read from a single windowed DFT.
**Primary diagnostic.** each window's main-lobe width (bins) vs peak side-lobe level (dB), read against a two-tone resolution test and a weak-neighbour visibility test.
**Transfer challenge.** the same trade decides whether you can see a low-amplitude harmonic beside a strong fundamental, or a sleep spindle beside an alpha peak — pick the window for the peak you must keep.

*Self-contained: seeded synthetic tones (`np.random.default_rng(2013)`), `scipy.signal` windows, no `bsp`, no I/O. Runs top-to-bottom in well under a minute. Theme (§1.8): there is **no single best** window — a narrow main lobe (resolution) and low side lobes (dynamic range) pull against each other — **but wrong choices still exist**, and the wrong window erases the very peak the question needs.*

### Companion lab · *Biomedical Signal Processing & Data Analytics* (CM2013)

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/farhad-abtahi/CM2013/blob/main/labs/lab04_windowing_leakage_choice/lab04_windowing_leakage_choice.ipynb) [![View](https://img.shields.io/badge/view-static-orange)](https://farhad-abtahi.github.io/CM2013/nb/lab04_windowing_leakage_choice.html) [![JupyterLite](https://img.shields.io/badge/run-JupyterLite-blue)](https://farhad-abtahi.github.io/CM2013/lite/lab/index.html?path=lab04_windowing_leakage_choice.ipynb)

In [ ]:
# --- shared setup (reproducible; self-contained, no bsp, no I/O) ---
import numpy as np, matplotlib.pyplot as plt
from scipy import signal as sig
rng = np.random.default_rng(2013)
plt.rcParams.update({"figure.dpi": 120, "axes.grid": True})

fs, N = 1000, 1000            # 1 s record  ->  frequency BIN = fs/N = 1.0 Hz
t = np.arange(N) / fs
BIN = fs / N                  # width of one DFT bin, in Hz (== 1.0 here, so bins == Hz)

# The three windows under test (periodic form, sym=False, the right choice for spectra).
WINDOWS = {
    "rectangular": np.ones(N),
    "Hann":        sig.windows.hann(N, sym=False),
    "Blackman":    sig.windows.blackman(N, sym=False),
}

# Heavy zero-padding does NOT add resolution (that is fixed by N and the window); it only
# INTERPOLATES the DTFT densely so we can SEE the main-lobe / side-lobe shape between DFT bins.
NFFT = 1 << 15                # 32768-point grid over the 1000-sample record

def spectrum(x, w, nfft=NFFT):
    """Windowed magnitude spectrum on a dense grid. Returns (freq_Hz, |X|)."""
    X = np.abs(np.fft.rfft(x * w, n=nfft))
    f = np.fft.rfftfreq(nfft, d=1 / fs)
    return f, X

def db(mag):
    """Magnitude -> dB, normalised so the largest value in the array is 0 dB."""
    return 20 * np.log10(mag / mag.max() + 1e-300)

# --- Signal A: two CLOSE equal-amplitude tones, 2 Hz (= 2 bins) apart ---
# Placed slightly OFF the exact DFT grid so the leakage we study is the generic case,
# not the special zero-leakage coincidence of an exactly-on-bin tone.
fA1, fA2 = 100.3, 102.3
xA = np.sin(2 * np.pi * fA1 * t) + np.sin(2 * np.pi * fA2 * t)

# --- Signal B: one STRONG tone + one WEAK tone 60 dB down, nearby, + light noise ---
f_strong, f_weak = 100.4, 120.7
a_weak = 10 ** (-60 / 20)                         # 60 dB below the strong tone -> 0.001
xB = (np.sin(2 * np.pi * f_strong * t)
      + a_weak * np.sin(2 * np.pi * f_weak * t)
      + 1e-4 * rng.standard_normal(N))            # light noise, well below the weak tone

print(f"fs={fs} Hz, N={N} samples, bin = fs/N = {BIN:.1f} Hz")
print(f"A: equal tones at {fA1} & {fA2} Hz  (separation {fA2 - fA1:.0f} Hz = {(fA2 - fA1) / BIN:.0f} bins)")
print(f"B: strong {f_strong} Hz + weak {f_weak} Hz at {20 * np.log10(a_weak):.0f} dB down + light noise")

## 1. Resolution — can the window tell two CLOSE equal tones apart?

Signal A holds two equal-amplitude tones **2 bins apart**. A window resolves them only if its **main lobe is narrow enough** to leave a dip between the two peaks. Compute A's magnitude spectrum with each window and apply a simple **two-peak (dip) test**: find the peak near each tone and the lowest point between them — they are *resolved* only if that valley sits at least `DIP_DB` below the weaker peak.

In [ ]:
# TODO for signal A, compute the windowed magnitude spectrum for EACH window, then decide
# whether the two tones are RESOLVED with a two-peak dip test: peak1 = the tallest point within
# +-0.7 Hz of fA1, peak2 = the tallest within +-0.7 Hz of fA2, and valley = the minimum of the
# dB curve between those two indices. dip = min(peak1, peak2) - valley (a merged pair gives
# dip ~ 0 or negative); resolved = dip >= DIP_DB. Fill resolvedA/dipA.
DIP_DB = 3.0
resolvedA, dipA = {}, {}
raise NotImplementedError("TODO: implement this — see the comment above")
# Checkpoint: which window resolves them, and does that match its main-lobe width (task 3)?

In [ ]:
# --- picture: signal A spectra, zoomed on the two tones (dotted = true tone frequencies) ---
plt.figure(figsize=(9, 3.2))
for name, w in WINDOWS.items():
    f, X = spectrum(xA, w)
    m = (f >= 96) & (f <= 107)
    plt.plot(f[m], db(X[m]), label=name)
for fc in (fA1, fA2):
    plt.axvline(fc, color="k", ls=":", lw=0.8)
plt.ylim(-60, 3); plt.xlabel("Hz"); plt.ylabel("dB (per window)")
plt.title("Signal A — narrow lobe (rectangular) splits the pair; wide lobe (Blackman) merges it")
plt.legend(); plt.show()

## 2. Leakage — do the window's side lobes BURY a weak neighbour?

Now the reverse failure. Signal B has a **strong** tone and, 20 bins away, a tone **60 dB weaker** plus light noise. A window hides the weak tone when its **side lobes** near the strong tone sit *above* the weak peak. Measure, for each window, the **weak-tone peak height above the local floor**: the tallest point within ±1 Hz of the weak tone, minus the median of the flanking band. Call it *revealed* only if that height clears `REVEAL_DB` **and** the peak actually lands on the weak tone (not on a side-lobe crest).

In [ ]:
# TODO for signal B, measure the weak tone's visibility for EACH window: peak = the tallest
# point (dB) within +-1 Hz of f_weak; floor = median of the dB curve over
# [f_weak-8, f_weak-3] U [f_weak+3, f_weak+8] Hz (the flanking bands); aboveB = peak - floor;
# peakfB = the frequency of that peak. Mark it REVEALED only if aboveB >= REVEAL_DB AND peakfB
# sits within 0.5 Hz of f_weak. Fill revealedB/aboveB/peakfB.
REVEAL_DB = 10.0
revealedB, aboveB, peakfB = {}, {}, {}
raise NotImplementedError("TODO: implement this — see the comment above")
# Checkpoint: the rectangular 'peak' is a side-lobe crest, not the weak tone -- why is that 'buried'?

In [ ]:
# --- picture: signal B spectra; can you see the weak tone at 120.7 Hz? ---
plt.figure(figsize=(9, 3.2))
for name, w in WINDOWS.items():
    f, X = spectrum(xB, w)
    m = (f >= 90) & (f <= 135)
    plt.plot(f[m], db(X[m]), label=name)
plt.axvline(f_weak, color="k", ls=":", lw=0.8)
plt.text(f_weak + 0.6, -55, "weak tone\n(-60 dB)", fontsize=8, va="top")
plt.ylim(-120, 3); plt.xlabel("Hz"); plt.ylabel("dB (rel. strong tone)")
plt.title("Signal B — rectangular side-lobe wall buries the weak tone; Blackman lets it through")
plt.legend(); plt.show()

## 3. The trade in numbers — main-lobe width vs peak side-lobe

The two failures are two ends of **one** trade, fixed by the window's own spectrum. Measure both **from each window alone** (no test signal): the **main-lobe width** null-to-null (in bins, = Hz here) and the **peak side-lobe level** (dB below the main lobe). A narrow main lobe buys **resolution** (task 1); low side lobes buy **dynamic range** for a weak neighbour (task 2). No window wins both — that is the whole lesson, made numeric.

In [ ]:
# TODO from EACH window's OWN spectrum (zero-padded), measure (a) the null-to-null main-lobe
# width in bins: walk outward from f=0 to the first local minimum (f_first_null), then
# width_bins = 2 * f_first_null / BIN (the lobe is symmetric about 0); and (b) the peak
# side-lobe level in dB: sll_db = max of the dB curve beyond that null. Fill
# mainlobe_bins/sll_db/first_null_hz, then print the trade table.
mainlobe_bins, sll_db, first_null_hz = {}, {}, {}
raise NotImplementedError("TODO: implement this — see the comment above")
# Checkpoint: narrow main lobe buys resolution; low side lobes buy dynamic range -- which window sits where?

In [ ]:
# --- picture of the trade: the three window spectra (main lobe + side lobes) ---
plt.figure(figsize=(9, 3.2))
for name, w in WINDOWS.items():
    W = np.abs(np.fft.rfft(w, n=NFFT))
    fW = np.fft.rfftfreq(NFFT, d=1 / fs)
    plt.plot(fW / BIN, 20 * np.log10(W / W.max() + 1e-300), label=name)
plt.xlim(0, 10); plt.ylim(-100, 3)
plt.xlabel("bins from centre (1 bin = fs/N = 1 Hz)"); plt.ylabel("dB")
plt.title("Window spectra — narrow lobe (rectangular) vs low side lobes (Blackman)")
plt.legend(); plt.show()

## 4. Live sanity check — the two failures are OPPOSITE, and are the trade

One block of assertions ties the lab together: the window that *wins* on A must *lose* on B, and that reversal must be exactly the main-lobe / side-lobe trade measured in task 3. All numbers below are computed live.

In [ ]:
# --- live sanity check: rectangular and Blackman fail on OPPOSITE signals (numbers computed live) ---
assert bool(resolvedA["rectangular"]), "rectangular should RESOLVE the two close tones"
assert not resolvedA["Blackman"], "Blackman's wide main lobe should MERGE the two close tones"
assert not revealedB["rectangular"], "rectangular's high side lobes should BURY the weak tone"
assert bool(revealedB["Blackman"]), "Blackman's low side lobes should REVEAL the weak tone"
# ...and that reversal IS the trade measured in task 3 (narrow lobe <-> high side lobes):
assert mainlobe_bins["rectangular"] < mainlobe_bins["Blackman"], "rect has the NARROWER main lobe"
assert sll_db["rectangular"]        > sll_db["Blackman"],        "rect has the HIGHER side lobes"
print("sanity check PASSED:")
print(f"  rectangular  resolves A (dip {dipA['rectangular']:5.1f} dB)  but BURIES B's weak tone "
      f"({aboveB['rectangular']:4.1f} dB above floor)")
print(f"  Blackman     merges A   (dip {dipA['Blackman']:5.1f} dB)  but REVEALS B's weak tone "
      f"({aboveB['Blackman']:4.1f} dB above floor)")
print(f"  the reversal is the trade: main lobe {mainlobe_bins['rectangular']:.1f} vs "
      f"{mainlobe_bins['Blackman']:.1f} bins, side lobes {sll_db['rectangular']:.0f} vs "
      f"{sll_db['Blackman']:.0f} dB")
print("  -> no single best window: the narrow lobe that resolves also has the high side lobes that bury.")

## Reflection

This reflection is for your own practice — there is nothing to submit. What matters is the *reasoning*, not naming a favourite window.

1. **Which failure did each window cause, and why?** Tie the rectangular window's success on A and failure on B to a single number from task 3 (its main-lobe width **and** its peak side-lobe level), and Blackman's opposite behaviour to the same two numbers.
2. **Read the trade.** Using the task-3 table, state the rule in one line: what does a *narrow* main lobe buy, and what does it cost? Where does **Hann** sit between the two extremes? Hann merges A (dip ≈ 0 dB, like Blackman) and reveals B (like Blackman) — so it is never the winner here. Using its task-3 numbers (4-bin lobe, -31 dB side lobes) explain both outcomes, and describe a signal for which Hann would be the right pick.
3. **Choose for the question.** For signal B (see a weak neighbour) which window would you pick, and for signal A (separate two equal close tones) which would you pick — and what single change to the **record** (a longer observation T) would let one window do both?

**Rule out.** Name one window that is wrong for one of the two signals, state which §1.8 'information that must be preserved' it destroys, and quote the task-3 number (main-lobe width or side-lobe level) that predicts the failure.

> *Your answers here.*

---
*Type-2 lab for **Biomedical Signal Processing & Data Analytics**. Self-contained synthetic tones; illustrative numbers. The lesson is the method, not the window: measure the main-lobe / side-lobe trade, then match the window to the peak you must preserve.*